# Adam vs the HAdam limits: dimension sweeps and an interesting covariance

This notebook produces four figures:

1. **Linear regression** — Adam simulator (80% CI over 20 seeds, no mean) for
   `d in {128, 256, 512}` against the d-independent HAdam ODE limit, identity
   covariance, `T=10`.
2. The same sweep for **logistic regression**.
3. **Adam vs Block Adam**, pointwise risk paths, for three (dimension, covariance)
   combinations.
4. For a fixed dimension and a dense ("interesting") covariance: the ODE limit
   together with 80% CI bands for the Adam simulator and the HAdam SDE.


In [ ]:
import config
config.enable_x64()

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

import problems, simulate, dynamics
from utils import compute_ci

BETA1, BETA2 = 0.1, 0.1
LR = 0.7          # continuous-time learning rate
N_SEEDS = 20
T = 10.0


def fixed_risk_init(cov, key, norm0=9.0, normstar=1.0):
    """theta0, theta* with prescribed Sigma-norms (fixes the initial risk)."""
    d = cov.shape[0]
    k0, ks = jax.random.split(key)
    theta0 = jax.random.normal(k0, (d, 1)) / jnp.sqrt(d)
    star = jax.random.normal(ks, (d, 1)) / jnp.sqrt(d)
    if cov.ndim == 1:
        n0 = theta0.T @ (cov[:, None] * theta0)
        ns = star.T @ (cov[:, None] * star)
    else:
        n0 = theta0.T @ cov @ theta0
        ns = star.T @ cov @ star
    theta0 = theta0 * jnp.sqrt(norm0 / n0)
    star = star * jnp.sqrt(normstar / ns)
    return theta0, star


def make_dense_cov(d, seed, power=0.7):
    spec = jnp.array([j ** -power for j in range(1, d + 1)])
    Qm, _ = jnp.linalg.qr(jax.random.normal(jax.random.PRNGKey(seed), (d, d)))
    K = (Qm * spec) @ Qm.T
    K = (K + K.T) / 2
    return K / (jnp.trace(K) / d)


## 1. Linear regression: Adam (80% CI) vs HAdam ODE across dimensions

In [ ]:
def sweep_figure(prob_name, dims=(128, 256, 512), ode_dim=512, fname=None):
    prob = problems.get_problem(prob_name)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    colors = plt.cm.viridis(jnp.linspace(0.15, 0.85, len(dims)))

    for color, d in zip(colors, dims):
        cov = jnp.ones(d)
        theta0, star = fixed_risk_init(cov, jax.random.PRNGKey(d))
        sim = simulate.build_adam(prob, cov, star, LR / d, beta1=BETA1, beta2=BETA2)
        keys = jax.random.split(jax.random.PRNGKey(100 + d), N_SEEDS)
        risks = simulate.run_many(sim, prob, theta0, star, cov, int(T * d), keys=keys)
        t = jnp.arange(int(T * d)) / d
        _, lo, hi = compute_ci(risks)
        ax.fill_between(t, lo, hi, color=color, alpha=0.4, label=f"Adam, d={d} (80% CI)")

    cov = jnp.ones(ode_dim)
    theta0, star = fixed_risk_init(cov, jax.random.PRNGKey(ode_dim))
    ode_risk, ode_t = dynamics.run_adam_ode(prob, theta0, star, cov, T, LR, beta1=BETA1, beta2=BETA2,
                                            dt=0.05, num_samples=5000, key=jax.random.PRNGKey(0))
    ax.plot(ode_t, ode_risk, "k--", lw=2.0, label="HAdam ODE limit")

    ax.set_xlabel("rescaled time  t = k / d")
    ax.set_ylabel("risk")
    ax.set_yscale("log")
    ax.set_title(f"Adam sim (80% CI, {N_SEEDS} seeds) vs ODE limit -- {prob_name}, identity covariance")
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    if fname:
        fig.savefig(fname, dpi=150)
    return fig


fig1 = sweep_figure("linreg", fname="figs/sweep_linreg.png")


## 2. Logistic regression: same sweep

In [ ]:
fig2 = sweep_figure("logreg", fname="figs/sweep_logreg.png")


## 3. Adam vs Block Adam: pointwise risk paths

In [ ]:
combos = [
    ("identity",   128, jnp.ones(128)),
    ("power-law",  256, jnp.array([j ** -0.5 for j in range(1, 257)]) / jnp.mean(jnp.array([j ** -0.5 for j in range(1, 257)]))),
    ("dense",      512, make_dense_cov(512, seed=9)),
]

prob = problems.get_problem("linreg")
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (name, d, cov) in zip(axes, combos):
    theta0, star = fixed_risk_init(cov, jax.random.PRNGKey(d))
    key = jax.random.PRNGKey(d)
    adam = simulate.build_adam(prob, cov, star, LR / d, beta1=BETA1, beta2=BETA2)
    badam = simulate.build_block_adam(prob, cov, star, LR / d, beta1=BETA1, beta2=BETA2)
    _, r_adam = simulate.run(adam, prob, theta0, star, cov, int(T * d), key=key)
    _, r_badam = simulate.run(badam, prob, theta0, star, cov, int(T * d), key=key)
    t = jnp.arange(int(T * d)) / d
    ax.plot(t, r_adam, ".", ms=2, alpha=0.6, label="Adam")
    ax.plot(t, r_badam, ".", ms=2, alpha=0.6, label="Block Adam")
    ax.set_yscale("log")
    ax.set_xlabel("t = k/d")
    ax.set_title(f"{name}, d={d}")

axes[0].set_ylabel("risk")
axes[0].legend(frameon=False, fontsize=9)
fig.suptitle("Adam vs Block Adam: pointwise risk paths")
fig.tight_layout()
fig.savefig("figs/adam_vs_block_adam_paths.png", dpi=150)


## 3b. Adam vs Resampled Adam: pointwise risk paths


In [ ]:
prob = problems.get_problem("linreg")
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (name, d, cov) in zip(axes, combos):
    theta0, star = fixed_risk_init(cov, jax.random.PRNGKey(d))
    key = jax.random.PRNGKey(d)
    adam = simulate.build_adam(prob, cov, star, LR / d, beta1=BETA1, beta2=BETA2)
    radam = simulate.build_resampled_adam(prob, cov, star, LR / d, beta1=BETA1, beta2=BETA2)
    _, r_adam = simulate.run(adam, prob, theta0, star, cov, int(T * d), key=key)
    _, r_radam = simulate.run(radam, prob, theta0, star, cov, int(T * d), key=key)
    t = jnp.arange(int(T * d)) / d
    ax.plot(t, r_adam, ".", ms=2, alpha=0.6, label="Adam")
    ax.plot(t, r_radam, ".", ms=2, alpha=0.6, label="Resampled Adam")
    ax.set_yscale("log")
    ax.set_xlabel("t = k/d")
    ax.set_title(f"{name}, d={d}")

axes[0].set_ylabel("risk")
axes[0].legend(frameon=False, fontsize=9)
fig.suptitle("Adam vs Resampled Adam: pointwise risk paths")
fig.tight_layout()
fig.savefig("figs/adam_vs_resampled_adam_paths.png", dpi=150)


## 4. ODE limit, Adam (80% CI), and HAdam SDE (80% CI) for a dense covariance

In [ ]:
d4 = 128
cov4 = make_dense_cov(d4, seed=21)
theta0_4, star_4 = fixed_risk_init(cov4, jax.random.PRNGKey(4242))
T4, LR4 = 3.0, 1.0

prob = problems.get_problem("linreg")

keys_adam = jax.random.split(jax.random.PRNGKey(1), N_SEEDS)
adam_risks = simulate.run_many(
    simulate.build_adam(prob, cov4, star_4, LR4 / d4, beta1=BETA1, beta2=BETA2),
    prob, theta0_4, star_4, cov4, int(T4 * d4), keys=keys_adam)
t_adam = jnp.arange(int(T4 * d4)) / d4
_, lo_adam, hi_adam = compute_ci(adam_risks)

ode_risk, ode_t = dynamics.run_adam_ode(prob, theta0_4, star_4, cov4, T4, LR4, beta1=BETA1, beta2=BETA2,
                                        dt=0.02, num_samples=8000, noise_samples=3000, noise_history=40,
                                        key=jax.random.PRNGKey(2))

keys_sde = jax.random.split(jax.random.PRNGKey(1000), N_SEEDS)
sde_runs, sde_t_all = jax.vmap(lambda k: dynamics.run_adam_sde(
    prob, theta0_4, star_4, cov4, T4, LR4, beta1=BETA1, beta2=BETA2,
    dt=0.01, num_samples=8000, noise_samples=3000, noise_history=40, key=k))(keys_sde)
sde_t = sde_t_all[0]
_, lo_sde, hi_sde = compute_ci(sde_runs)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.fill_between(t_adam, lo_adam, hi_adam, color="C0", alpha=0.35, label=f"Adam ({N_SEEDS} seeds, 80% CI)")
ax.fill_between(sde_t, lo_sde, hi_sde, color="C1", alpha=0.35, label=f"HAdam SDE ({N_SEEDS} seeds, 80% CI)")
ax.plot(ode_t, ode_risk, "k--", lw=2.0, label="HAdam ODE")
ax.set_xlabel("t = k/d")
ax.set_ylabel("risk")
ax.set_yscale("log")
ax.set_title(f"d={d4}, dense covariance (power-law spectrum, random rotation)")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig("figs/ode_adam_sde_dense.png", dpi=150)
